# Deploying EmberX v2 (YOLO11s) using Roboflow Inference SDK on Google Colab

This notebook focuses on **Method 1: Local Inference via Roboflow `inference` SDK** to deploy your fire, smoke, and human detector model (`fire-smoke-and-human-detector-cokiv/1`) onto Google Colab.

## Quick Steps:
1. **Enable GPU (Optional but recommended for videos)**: Click on **Runtime** -> **Change runtime type** -> select **T4 GPU** -> Save.
2. **Get your API Key**: Copy your private API key from your Roboflow Workspace under Settings -> Developer.

### 1. Install Dependencies
We need to install `inference` (Roboflow local server), `supervision` (for annotations), and `opencv-python`.

In [ ]:
!pip install -q inference supervision opencv-python pillow

### 2. Securely Set Roboflow API Key
This cell securely prompts you for your API key so it is not hardcoded in your notebook.

In [ ]:
import os
from getpass import getpass

if "ROBOFLOW_API_KEY" not in os.environ:
    os.environ["ROBOFLOW_API_KEY"] = getpass("Enter your Roboflow Private API Key: ")

### 3. Initialize the Model
We load your exact model trained on Roboflow (`fire-smoke-and-human-detector-cokiv/1`). The model weights will automatically be downloaded and cached locally.

In [ ]:
from inference import get_model

model_id = "fire-smoke-and-human-detector-cokiv/1"
model = get_model(model_id=model_id)
print(f"Model '{model_id}' successfully loaded!")

### 4. Run Inference on a Test Image (URL)
Test the model on a sample web image to verify everything works.

In [ ]:
import cv2
import supervision as sv
from PIL import Image

# 1. Select a test image URL (a sample fire/smoke or human image)
image_url = "https://media.roboflow.com/notebooks/examples/dog.jpeg"

# 2. Run inference
results = model.infer(image_url)[0]

# 3. Map predictions to supervision detections
detections = sv.Detections.from_inference(results)

# 4. Annotate image
image = sv.download_image(image_url)
box_annotator = sv.BoundingBoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

# 5. Display result
display(Image.fromarray(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)))

### 5. Run Inference on your Own Uploaded Image
Use Colab's file upload utility to test a local image of yours.

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

if uploaded_files:
    target_image = uploaded_files[0]
    print(f"Processing: {target_image}")

    # Run inference
    image = cv2.imread(target_image)
    results = model.infer(image)[0]
    detections = sv.Detections.from_inference(results)

    # Annotate
    box_annotator = sv.BoundingBoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
    annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

    # Display
    display(Image.fromarray(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)))
else:
    print("No image uploaded.")

### 6. Run Inference on a Video (T4 GPU Recommended)
Upload a video file to run inference frame-by-frame and compile it back into an annotated video file you can download.

In [ ]:
import cv2
import supervision as sv
from google.colab import files

print("Upload your test video:")
uploaded_video = files.upload()
video_keys = list(uploaded_video.keys())

if video_keys:
    input_video = video_keys[0]
    output_video = "annotated_" + input_video
    print(f"Processing video: {input_video} -> saving output to: {output_video}")

    # Initialize supervision annotators
    box_annotator = sv.BoundingBoxAnnotator()
    label_annotator = sv.LabelAnnotator()

    # Define callback function to process each frame
    def callback(frame: cv2.Mat, frame_index: int) -> cv2.Mat:
        results = model.infer(frame)[0]
        detections = sv.Detections.from_inference(results)
        
        annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
        annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections)
        return annotated_frame

    # Run the frame processing pipeline
    sv.process_video(
        source_path=input_video,
        target_path=output_video,
        callback=callback
    )
    print("Processing complete! Downloading annotated video...")
    files.download(output_video)
else:
    print("No video uploaded.")